# CAY Replication Tour

This notebook inspects the processed quarterly panel and generated report artifacts. It does not reacquire data or reimplement the estimation pipeline.

In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "reports" / "report_contract.yml").exists():
            return candidate
    raise FileNotFoundError("Could not locate reports/report_contract.yml from the notebook path.")


ROOT = find_repository_root(Path.cwd().resolve())
PANEL_PATH = ROOT / "_data" / "processed" / "core_quarterly.parquet"
AUDIT_PATH = ROOT / "reports" / "tables" / "table_r1_replication_audit.csv"
METADATA_PATH = ROOT / "reports" / "build" / "report_metadata.json"

if not PANEL_PATH.exists():
    raise FileNotFoundError(f"Missing processed panel: {PANEL_PATH}. Run `python -m src.pipeline panel`.")

panel = pd.read_parquet(PANEL_PATH)
if not isinstance(panel.index, pd.PeriodIndex):
    panel.index = pd.PeriodIndex(panel.index, freq="Q")
panel = panel.sort_index()
print(f"Repository: {ROOT}")
print(f"Panel: {panel.index.min()} through {panel.index.max()} ({len(panel)} quarters)")

FileNotFoundError: Missing processed panel: /Users/zhichengzhou/Google Drive/UChicago/FINM329/hw/Study_of_Cay_and_Predictivity_Lab/_data/processed/core_quarterly.parquet. Run `python -m src.pipeline panel`.

## Coverage

Inspect the first and last observed quarter and missing counts for the macro levels, independently estimated CAY, and both excess-return series.

In [ ]:
coverage_columns = ["c", "a", "y", "cay", "sp_excess_return", "crsp_vw_excess_return"]
missing_columns = sorted(set(coverage_columns) - set(panel.columns))
if missing_columns:
    raise ValueError(f"Processed panel lacks notebook columns: {missing_columns}. Run `python -m src.pipeline exhibits`.")

coverage = pd.DataFrame(
    {
        "first_quarter": [str(panel[column].first_valid_index()) for column in coverage_columns],
        "last_quarter": [str(panel[column].last_valid_index()) for column in coverage_columns],
        "observations": [int(panel[column].count()) for column in coverage_columns],
        "missing": [int(panel[column].isna().sum()) for column in coverage_columns],
    },
    index=coverage_columns,
)
coverage

## Summary Statistics

The table reports sample size, moments, percentiles, and first-order autocorrelation from the processed panel.

In [ ]:
summary_rows = []
for column in coverage_columns:
    series = panel[column].dropna()
    summary_rows.append(
        {
            "variable": column,
            "observations": len(series),
            "mean": series.mean(),
            "standard_deviation": series.std(ddof=1),
            "p05": series.quantile(0.05),
            "median": series.median(),
            "p95": series.quantile(0.95),
            "ar1": series.corr(series.shift(1)),
        }
    )
summary = pd.DataFrame(summary_rows).set_index("variable")
summary

## Data Anatomy

The first panel indexes real per-capita macro levels to 100 at their first common observation. The second standardizes CAY and S&P excess returns over their displayed common sample.

In [ ]:
macro = panel[["c", "y", "a"]].dropna()
indexed = 100 * np.exp(macro - macro.iloc[0])
descriptive = panel[["cay", "sp_excess_return"]].dropna()
standardized = (descriptive - descriptive.mean()) / descriptive.std(ddof=1)

figure, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)
indexed.rename(columns={"c": "consumption", "y": "labor income", "a": "net worth"}).plot(ax=axes[0])
axes[0].set_ylabel("Index = 100")
axes[0].set_title("Real per-capita macro levels")
standardized.plot(ax=axes[1])
axes[1].set_ylabel("Standard deviations")
axes[1].set_title("CAY and S&P excess returns")
for axis in axes:
    axis.spines[["top", "right"]].set_visible(False)
figure.tight_layout()
plt.show()

## Replication Audit

The generated audit distinguishes strict historical matches, revised-vintage matches, and checks that still require diagnosis.

In [ ]:
if not AUDIT_PATH.exists():
    raise FileNotFoundError(f"Missing replication audit: {AUDIT_PATH}. Run `python -m src.pipeline exhibits`.")
audit = pd.read_csv(AUDIT_PATH)
audit

## Build Metadata

The final generated metadata records sample endpoints, data vintage, CAY definitions, selected rate conventions, and Git revision.

In [ ]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Missing report metadata: {METADATA_PATH}. Run `python -m src.pipeline exhibits`.")
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
pd.Series(metadata, name="value").to_frame()